In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import models, layers

In [2]:
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

In [3]:
from pathlib import Path
train_ds, val_ds = tf.keras.utils.audio_dataset_from_directory(
    directory=str(Path.home() / ".cache" / "ml-notebooks" / "speech-commands"),
    seed=seed,
    batch_size=64, 
    output_sequence_length=16000,
    validation_split=0.3,
    subset='both'
)

Found 6505 files belonging to 4 classes.


Using 4554 files for training.


Using 1951 files for validation.


In [4]:
label = np.array(train_ds.class_names)
# print(label)

In [5]:
def squeeze(audio, label):
    audio = tf.squeeze(audio, axis=-1)
    return audio, label

def get_st(wf):
    st = tf.signal.stft(wf, 255, 128)
    st = tf.abs(st)
    st = st[..., tf.newaxis]
    return st

def make_st_ds(ds):
    return ds.map(
        lambda i, j: (get_st(i), j),
        tf.data.AUTOTUNE
    )

In [6]:
train_ds = train_ds.map(squeeze, tf.data.AUTOTUNE)
val_ds = val_ds.map(squeeze, tf.data.AUTOTUNE)

test_ds = val_ds.shard(num_shards=2, index=0)
val_ds = val_ds.shard(num_shards=2, index=1)

In [7]:
train_st_ds = make_st_ds(train_ds)
val_st_ds = make_st_ds(val_ds)
test_st_ds = make_st_ds(test_ds)

In [8]:
for i, j in train_st_ds.take(1):
    break

input_shape = i.shape[1:]
label_len = len(label)

# print(input_shape)
# print(label_len)

In [9]:
norm_layer = layers.Normalization()
norm_layer.adapt(
    data=train_st_ds.map(
        lambda i, j: i
    )
)

In [10]:
model = models.Sequential([
    layers.Input(input_shape),
    layers.Resizing(32, 32),
    norm_layer,
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.5),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(label_len)
])

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ (None, 32, 32, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization (Normalization)   │ (None, 32, 32, 1)      │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 30, 30, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,255,431 (4.79 MB)

 Trainable params: 1,255,428 (4.79 MB)

 Non-trainable params: 3 (16.00 B)

In [12]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [13]:
history = model.fit(
    train_st_ds,
    validation_data=val_st_ds,
    epochs=10,
    callbacks=tf.keras.callbacks.EarlyStopping(verbose=1, patience=2)
)

Epoch 1/10


C:\Users\jorda\Code\ml-notebooks\courses\speech-recognition\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


 1/72 ━━━━━━━━━━━━━━━━━━━━ 57s 809ms/step - accuracy: 0.2031 - loss: 1.4288

 3/72 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.2969 - loss: 1.4134  

 5/72 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.3313 - loss: 1.3606

 7/72 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.3482 - loss: 1.3268

... (302 outputs left out)

65/72 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9353 - loss: 0.1844

68/72 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9359 - loss: 0.1826

71/72 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9368 - loss: 0.1799

72/72 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.9370 - loss: 0.1796 - val_accuracy: 0.9417 - val_loss: 0.1632


In [14]:
for i, (spec_batch, lab_batch) in enumerate(test_st_ds):
    pred = model.predict(spec_batch)

    for j in range(len(lab_batch)):
        t = lab_batch[j].numpy()
        p = np.argmax(pred[j])

        print(f'{t}, {p}')

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step 


2, 2
1, 1
2, 2
0, 0
2, 2
0, 0
0, 0
0, 0
0, 0
1, 1
3, 3
1, 1
2, 2
0, 0
0, 0
2, 2
1, 2
0, 0
3, 3
1, 1
1, 1
2, 2
1, 1
1, 1
3, 3
2, 2
2, 2
0, 0
0, 0
1, 1
0, 0
3, 3
3, 3
1, 1
1, 1
0, 0
0, 0
3, 3
1, 1
2, 2
3, 3
2, 2
2, 2
3, 3
1, 1
2, 2
3, 3
1, 1
0, 0
1, 1
2, 2
3, 3
2, 2
2, 2
0, 0
3, 3
3, 3
2, 2
0, 0
3, 3
2, 2
0, 0
1, 1
1, 1
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


3, 3
2, 2
1, 1
2, 2
0, 0
3, 3
0, 0
2, 2
3, 3
3, 3
3, 3
0, 0
0, 0
1, 1
1, 1
0, 0
3, 1
0, 0
3, 3
1, 1
2, 2
0, 0
0, 0
0, 0
1, 1
1, 3
3, 3
1, 1
0, 0
0, 0
3, 3
0, 0
2, 2
0, 0
1, 1
2, 2
1, 1
1, 1
0, 2
1, 1
2, 2
2, 2
1, 1
2, 2
1, 1
0, 0
2, 2
0, 0
3, 3
1, 1
0, 0
3, 3
1, 1
1, 1
0, 0
3, 3
0, 0
0, 0
2, 2
3, 3
1, 1
3, 3
0, 0
0, 0
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


... (40 outputs left out)

3, 3
0, 0
1, 1
0, 2
0, 0
3, 3
3, 3
0, 0
2, 2
0, 0
2, 2
0, 0
0, 0
2, 2
1, 1
3, 3
3, 3
0, 0
1, 1
3, 3
1, 1
3, 3
0, 0
3, 3
2, 3
1, 2
2, 2
1, 1
2, 2
3, 3
0, 0
3, 2
1, 1
3, 2
1, 1
2, 2
3, 3
2, 2
2, 2
3, 3
0, 0
3, 2
3, 3
0, 0
1, 1
1, 1
3, 3
3, 3
1, 2
0, 0
2, 2
1, 1
3, 3
0, 0
0, 0
3, 3
2, 2
3, 3
2, 2
2, 2
0, 0
0, 0
0, 0


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step


2, 2
3, 2
2, 2
2, 2
2, 2
0, 0
3, 3
3, 3
3, 3
1, 1
3, 3
1, 1
0, 0
1, 1
0, 0
1, 1
2, 2
2, 2
3, 3
3, 2
2, 2
3, 3
0, 0
3, 3
3, 3
1, 1
2, 2
3, 3
3, 3
3, 3
0, 2
